# 01 - Agrégation, nettoyage et consolidation des données

**Objectif :** construire un jeu de données unique, à l'échelle de l'**EHPAD** (établissement), pour **prédire le tarif mensuel d'hébergement permanent en chambre seule** (`PRIX_MOIS_HEBPERM_CS_GIR56`). Les caractéristiques du territoire sont calculées soit à l'**échelle de la commune**, soit à l'**échelle de l'EPCI**, qui correspond mieux au bassin de vie et limite les problèmes de petits effectifs et de secret statistique. La partie **02_exploration** permettra de déterminer les variables définitives nécessaires au modèle.

**Jeux de données utilisés :**

- **[`cnsa-export-prix-ehpad2019.xlsx`](https://www.data.gouv.fr/datasets/prix-hebergement-et-tarifs-dependance-des-ehpad) (CNSA)** : la variable cible (tarif d'hébergement permanent), le nombre d'établissements du groupe, plus le statut juridique, l'habilitation à l'aide sociale (ASH) et le nombre de places.
- **[`etablab_stock_et_2019.csv`](https://static.data.gouv.fr/resources/finess-extraction-des-equipements-sociaux-et-medico-sociaux/20260108-154707/etalab-stock-equipsoc-historique-2004-2025.zip) (FINESS)** : le code commune de chaque EHPAD, pour le situer sur le territoire. Contient également plus de détails sur ce qui intervient dans la définition du prix, qui pourront être utilisés plus tard.
- **[`epcicom2020`](https://www.collectivites-locales.gouv.fr/files/files/Etudes-et-statistiques/Liste%20et%20composition%20des%20EPCI%20%C3%A0%20fiscalit%C3%A9%20propre/2020/epcicom2020.xlsx)** : la correspondance entre communes et EPCI, pour passer à l'échelle intercommunale. Le découpage 2020 a été choisi car c'est celui sur lequel se basent les données Filosofi 2019.
- **[`base-cc-evolution-structure`](https://www.insee.fr/fr/statistiques/fichier/6456153/base-ccc-evol-struct-pop-2019.zip) (INSEE, recensement 2019)** : la part des 75 ans et plus dans la population de l'EPCI, qui reflète la demande potentielle.
- **[`FILO2019_DISP_EPCI.csv`](https://www.insee.fr/fr/statistiques/6036907) (INSEE Filosofi 2019)** : le revenu médian et les indicateurs par âge (patrimoine, prestations sociales), qui reflètent le niveau de vie local.
- **[`indicateurs-loyers-appartements.csv`](https://static.data.gouv.fr/resources/carte-des-loyers-indicateurs-de-loyers-dannonce-par-commune-en-2018/20201203-114600/indicateurs-loyers-appartements.csv) (Carte des loyers, 2018)** : le loyer prédit au m², pondéré par la population, comme indicateur du coût de l'immobilier local.

**Sortie attendue :** un DataFrame à l'échelle de l'établissement qui servira à l'exploration (`02_exploration`) puis à la modélisation (`03_modelisation`).

In [63]:
#Bibliothèques
import pandas as pd

In [64]:
# Import des données base-cc-evol-struct-pop-2019.xlsx avec les id communes, reg, dep et population
df_communes = pd.read_excel('../data/raw/base-cc-evol-struct-pop-2019.xlsx',
                    sheet_name='COM_2019',
                    skiprows=5,
                    header=0
                    )
print(df_communes.columns.tolist())

['CODGEO', 'REG', 'DEP', 'LIBGEO', 'P19_POP', 'P19_POP0014', 'P19_POP1529', 'P19_POP3044', 'P19_POP4559', 'P19_POP6074', 'P19_POP7589', 'P19_POP90P', 'P19_POPH', 'P19_H0014', 'P19_H1529', 'P19_H3044', 'P19_H4559', 'P19_H6074', 'P19_H7589', 'P19_H90P', 'P19_H0019', 'P19_H2064', 'P19_H65P', 'P19_POPF', 'P19_F0014', 'P19_F1529', 'P19_F3044', 'P19_F4559', 'P19_F6074', 'P19_F7589', 'P19_F90P', 'P19_F0019', 'P19_F2064', 'P19_F65P', 'P19_POP01P', 'P19_POP01P_IRAN1', 'P19_POP01P_IRAN2', 'P19_POP01P_IRAN3', 'P19_POP01P_IRAN4', 'P19_POP01P_IRAN5', 'P19_POP01P_IRAN6', 'P19_POP01P_IRAN7', 'P19_POP0114_IRAN2P', 'P19_POP0114_IRAN2', 'P19_POP0114_IRAN3P', 'P19_POP1524_IRAN2P', 'P19_POP1524_IRAN2', 'P19_POP1524_IRAN3P', 'P19_POP2554_IRAN2P', 'P19_POP2554_IRAN2', 'P19_POP2554_IRAN3P', 'P19_POP55P_IRAN2P', 'P19_POP55P_IRAN2', 'P19_POP55P_IRAN3P', 'C19_POP15P', 'C19_POP15P_CS1', 'C19_POP15P_CS2', 'C19_POP15P_CS3', 'C19_POP15P_CS4', 'C19_POP15P_CS5', 'C19_POP15P_CS6', 'C19_POP15P_CS7', 'C19_POP15P_CS8', '

In [65]:
#Conservation des colonnes utiles
df_communes = df_communes[['CODGEO', 'REG', 'DEP', 'P19_POP', 'P19_POP7589', 'P19_POP90P']]

#Création d'une variable unique pour les +75 ans et suppression des colonnes utilisées
df_communes['P19_POP75P'] = df_communes['P19_POP7589'] + df_communes['P19_POP90P']
df_communes = df_communes.drop(axis='columns', columns =['P19_POP7589', 'P19_POP90P'])

df_communes.head()

,CODGEO,REG,DEP,P19_POP,P19_POP75P
0,01001,84,01,779,61.336604
1,01002,84,01,256,19.649743
2,01004,84,01,14134,1239.164308
3,01005,84,01,1751,120.860269
4,01006,84,01,112,11.200000


> Les arrondissements pour Paris, Lyon et Marseille sont dans une feuille distincte.
> Les établissements sont rattachés au code géo des arrondissements sur lesquels ils sont implantés et non au niveau général

In [66]:
# Import du DataFrame avec les id communes pour les arrondissements Marseille, Lyon et Paris
df_arr = pd.read_excel('../data/raw/base-cc-evol-struct-pop-2019.xlsx',
                    sheet_name='ARM_2019',
                    skiprows=5,
                    header=0
                    )

#Conservation des colonnes utiles
df_arr = df_arr[['CODGEO', 'REG', 'DEP', 'P19_POP', 'P19_POP7589', 'P19_POP90P']]

#Création d'une variable unique pour les +75 ans et suppression des colonnes utilisées
df_arr['P19_POP75P'] = df_arr['P19_POP7589'] + df_arr['P19_POP90P']
df_arr = df_arr.drop(axis='columns', columns =['P19_POP7589', 'P19_POP90P'])

df_arr

,CODGEO,REG,DEP,P19_POP,P19_POP75P
0,13201,93,13,39893,2822.460249
1,13202,93,13,25217,2273.435515
2,13203,93,13,51792,3470.640341
3,13204,93,13,49296,5572.715736
4,13205,93,13,46152,3683.766693
5,13206,93,13,42135,3867.505585
6,13207,93,13,34697,4599.868129
7,13208,93,13,81436,11194.724319
8,13209,93,13,76633,9055.474203
9,13210,93,13,58491,5431.218684


In [67]:
print(f"Longueur du dataframe avant la concatenation : {len(df_communes)}")
df_communes = pd.concat([df_communes, df_arr], ignore_index=True)
print(f"Longueur du dataframe après la concatenation : {len(df_communes)}")

#Pas de suppression immédiate des lignes des communes mère pour être sûr que l'aggrégation par EPCI soit OK

Longueur du dataframe avant la concatenation : 34938
Longueur du dataframe après la concatenation : 34983


In [68]:
# Import du dataframe contenant les codes EPCI
df_insee_epci = pd.read_excel('../data/raw/epcicom2020.xlsx',
                    header=0,
                    dtype={"siren": str} #pour anticiper la concat
                    )

print(df_insee_epci.columns.tolist())

['dept', 'siren', 'raison_sociale', 'nature_juridique', 'mode_financ', 'nb_membres', 'total_pop_tot', 'total_pop_mun', 'dep_com', 'insee', 'siren_membre', 'nom_membre', 'ptot_2020', 'pmun_2020']


In [69]:
#Conservation des colonnes utiles à la jointure
df_insee_epci = df_insee_epci[['siren','insee']]

#Jointure directement sur le df_communes
df_communes = pd.merge(left=df_communes,
            right=df_insee_epci,
            how='left', 
            left_on='CODGEO', 
            right_on='insee',
            validate='many_to_one').drop(columns="insee")#suppression de la colonne en double

In [70]:
df_arr.info()

<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CODGEO      45 non-null     int64  
 1   REG         45 non-null     int64  
 2   DEP         45 non-null     int64  
 3   P19_POP     45 non-null     int64  
 4   P19_POP75P  45 non-null     float64
dtypes: float64(1), int64(4)
memory usage: 1.9 KB


In [71]:
# Transformation de la colonne de int en str
df_communes['CODGEO'] = df_communes['CODGEO'].astype(str)

# Les codes INSEE ne sont pas associés à leurs EPCI
# Construction d'un dictionnaire pour associer chaque arrondissement à sa commune
# range(a, b) génère les codes des arrondissements
# ** pour déballer les trois sous dictionnaires dan codes_arr
codes_arr = {
    **{str(c): '13055' for c in range(13201, 13217)},  # 13055 = Marseille
    **{str(c): '69123' for c in range(69381, 69390)},  # 69123 = Lyon
    **{str(c): '75056' for c in range(75101, 75121)},  # 75056 = Paris
}

# Pour retrouver l'EPCI de chaque arrondissement
# Transformation de df_communes en « dictionnaire » : CODGEO -> siren de l'EPCI
siren_meres = df_communes.set_index('CODGEO')['siren']

# Attribution aux arrondissements l'EPCI de leur ville
# mask_arr vaut True sur les lignes dont le CODGEO est un arrondissement
mask_arr = df_communes['CODGEO'].isin(codes_arr)

# Sur ces lignes uniquement, on fait deux traductions successives :
#   code arrondissement --(codes_arr)--> code ville --(siren_meres)--> siren de l'EPCI
# puis on écrit le résultat dans la colonne 'siren'.
df_communes.loc[mask_arr, 'siren'] = (
    df_communes.loc[mask_arr, 'CODGEO'].map(codes_arr).map(siren_meres)
)

#Vérification
print(mask_arr.sum())                      # attendu : 45
print(df_communes['siren'].isnull().sum())  # attendu : 6

45
5


In [72]:
df_communes[mask_arr]

,CODGEO,REG,DEP,P19_POP,P19_POP75P,siren
34938,13201,93,13,39893,2822.460249,200054807
34939,13202,93,13,25217,2273.435515,200054807
34940,13203,93,13,51792,3470.640341,200054807
34941,13204,93,13,49296,5572.715736,200054807
34942,13205,93,13,46152,3683.766693,200054807
34943,13206,93,13,42135,3867.505585,200054807
34944,13207,93,13,34697,4599.868129,200054807
34945,13208,93,13,81436,11194.724319,200054807
34946,13209,93,13,76633,9055.474203,200054807
34947,13210,93,13,58491,5431.218684,200054807


In [73]:
# On travaille sur une copie pour ne pas modifier df_communes pendant la vérification.
df = df_communes.copy()

# CODGEO en str
df['CODGEO'] = df['CODGEO'].astype(str)


# codes_arr.values() contient les codes des villes ('13055', '69123', '75056'), répétés pour chaque arrondissement
# set(...) supprime les doublons et ne garde que ces 3 codes
# On garde les lignes de ces 3 villes, on met CODGEO en index et on récupère leur population
pop_meres = df[df['CODGEO'].isin(set(codes_arr.values()))].set_index('CODGEO')['P19_POP']


# On garde uniquement les lignes des arrondissements (les clés de codes_arr).
arr = df[df['CODGEO'].isin(codes_arr)]

# Chaque code d'arrondissement est remplacé par le code de sa ville (13201 -> 13055, etc.),
# puis on additionne les populations par ville.
pop_arr = arr.groupby(arr['CODGEO'].map(codes_arr))['P19_POP'].sum()


# Une ligne par ville : population officielle, somme des arrondissements, et différence entre les deux.
check = pd.DataFrame({'pop_commune_mere': pop_meres, 'somme_arrondissements': pop_arr})
check['ecart'] = check['pop_commune_mere'] - check['somme_arrondissements']
display(check)


# assert arrête le notebook avec le message d'erreur si la condition est fausse.
assert len(arr) == 45, f"{len(arr)} arrondissements trouvés au lieu de 45"   # 16 + 9 + 20
assert (check['ecart'] == 0).all(), "Écart entre communes mères et arrondissements"  # aucun écart sur les 3 villes
print("OK : somme des arrondissements = population des communes mères")

,pop_commune_mere,somme_arrondissements,ecart
CODGEO,,,
13055,870731,870731,0
69123,522969,522969,0
75056,2165423,2165423,0


OK : somme des arrondissements = population des communes mères


In [74]:
codes_meres = ['13055', '69123', '75056'] #Marseille, Lyon et Paris

# On supprime les lignes de ces trois villes pour ne garder que leurs arrondissements, sinon leur population serait comptée deux fois (ville entière + somme des arrondissements)

df_communes = df_communes[~df_communes['CODGEO'].isin(codes_meres)].reset_index(drop=True)

# Vérification : il ne doit plus rester aucune des trois villes
print(df_communes['CODGEO'].isin(codes_meres).sum())  # attendu : 0

print(len(df_communes))

0
34980


In [75]:
#Identification des siren manquantes
mask_empty_siren = df_communes['siren'].isnull()
df_communes[mask_empty_siren]

,CODGEO,REG,DEP,P19_POP,P19_POP75P,siren
7288,22016,53,22,352,69.648940,NaN
9368,27058,28,27,1769,257.111343,NaN
10345,29083,53,29,260,46.000000,NaN
10407,29155,53,29,833,163.000000,NaN
32667,85113,52,85,4850,712.036322,NaN


| Code commune | Commune | Situation | EHPAD / remarque |
|---|---|---|---|
| 22016 | Île-de-Bréhat | Pas d'intercommunalité | 1 EHPAD à retirer du dataset (FINESS 220013064) |
| 27058 | Bernières-sur-Seine | Commune déléguée | OK |
| 29083 | Île-de-Sein | Pas d'intercommunalité | Pas d'EHPAD |
| 29155 | Ouessant | Pas d'intercommunalité | 1 EHPAD (FINESS 290023571) |
| 85113 | Île-d'Yeu | Pas d'intercommunalité | 2 EHPAD (FINESS 850005034, 850003179) |

In [76]:
finess_to_delete = ["220013064", "290023571", "850005034", "850003179"] 
#Issu d'une recherche manuelle, à voir si je peux l'automatiser + tard (en str car certains finess ont p-ê lettre comme la corse)

In [77]:
#Chargement du dataset contenant le prix des loyers par commune
df_loyer = pd.read_csv("../data/raw/indicateurs-loyers-appartements.csv", sep=";", encoding="latin-1")
display(df_loyer.head())

print(df_loyer.info())

,id_zone,INSEE,LIBGEO,DEP,REG,EPCI,TYPPRED,loypredm2,lwr.IPm2,upr.IPm2,R2adj,NBobs_maille,NBobs_commune
0,2362,1001,L'Abergement-Clémenciat,1,84,200069193,maille,"9,372335348","7,409663525","11,85487972","0,774249278",701,2
1,2010,1002,L'Abergement-de-Varey,1,84,240100883,maille,"8,63555202","6,72107751","11,09535764","0,719598662",861,0
2,1456,1004,Ambérieu-en-Bugey,1,84,240100883,commune,"10,07450708","7,865307059","12,90422511","0,670160882",2156,2156
3,2362,1005,Ambérieux-en-Dombes,1,84,200042497,maille,"9,372335348","7,409663525","11,85487972","0,774249278",701,81
4,2055,1006,Ambléon,1,84,200040350,maille,"8,96695486","7,101593346","11,3222872","0,752823051",842,3


<class 'pandas.DataFrame'>
RangeIndex: 35441 entries, 0 to 35440
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_zone        35441 non-null  str  
 1   INSEE          35441 non-null  str  
 2   LIBGEO         35441 non-null  str  
 3   DEP            35441 non-null  str  
 4   REG            35441 non-null  int64
 5   EPCI           35441 non-null  str  
 6   TYPPRED        35441 non-null  str  
 7   loypredm2      35441 non-null  str  
 8   lwr.IPm2       35441 non-null  str  
 9   upr.IPm2       35441 non-null  str  
 10  R2adj          35441 non-null  str  
 11  NBobs_maille   35441 non-null  int64
 12  NBobs_commune  35441 non-null  int64
dtypes: int64(3), str(10)
memory usage: 3.5 MB
None


In [78]:
# Ajout d'un 0 devant les codes INSEE à 4 chiffres (dpt 01 à 09)
df_loyer["INSEE"] = df_loyer["INSEE"].str.zfill(5)
df_loyer.head(2)

,id_zone,INSEE,LIBGEO,DEP,REG,EPCI,TYPPRED,loypredm2,lwr.IPm2,upr.IPm2,R2adj,NBobs_maille,NBobs_commune
0,2362,01001,L'Abergement-Clémenciat,1,84,200069193,maille,"9,372335348","7,409663525","11,85487972","0,774249278",701,2
1,2010,01002,L'Abergement-de-Varey,1,84,240100883,maille,"8,63555202","6,72107751","11,09535764","0,719598662",861,0


In [79]:
#Conservation des colonnes utiles pour la jointure
df_loyer = df_loyer[['INSEE', 'loypredm2']]

#Jointure directement effectué sur le df_communes
df_communes = pd.merge(left = df_communes,
               right = df_loyer, 
               left_on = 'CODGEO', 
               right_on = 'INSEE',
               how = 'left',
               validate='one_to_one').drop(columns="INSEE")
df_communes.head(2)

,CODGEO,REG,DEP,P19_POP,P19_POP75P,siren,loypredm2
0,01001,84,01,779,61.336604,200069193,"9,372335348"
1,01002,84,01,256,19.649743,240100883,"8,63555202"


In [80]:
#Identification des valeurs manquantes
mask_empty_loy = df_communes['loypredm2'].isnull()
df_communes[mask_empty_loy]

,CODGEO,REG,DEP,P19_POP,P19_POP75P,siren,loypredm2
4867,14666,28,14,1906,138.304175,200065597,NaN
9368,27058,28,27,1769,257.111343,NaN,NaN


*  14666 et 27058, ce sont les communes délégués : les loyers doivent se trouver dans la commune rattaché

In [81]:
df_communes.head()

,CODGEO,REG,DEP,P19_POP,P19_POP75P,siren,loypredm2
0,01001,84,01,779,61.336604,200069193,"9,372335348"
1,01002,84,01,256,19.649743,240100883,"8,63555202"
2,01004,84,01,14134,1239.164308,240100883,"10,07450708"
3,01005,84,01,1751,120.860269,200042497,"9,372335348"
4,01006,84,01,112,11.200000,200040350,"8,96695486"


In [82]:
#Passage de loypredm2 en float
df_communes['loypredm2'] = df_communes['loypredm2'].astype(str).str.replace(',', '.').astype(float)
df_communes.info()

<class 'pandas.DataFrame'>
RangeIndex: 34980 entries, 0 to 34979
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CODGEO      34980 non-null  str    
 1   REG         34980 non-null  int64  
 2   DEP         34980 non-null  object 
 3   P19_POP     34980 non-null  int64  
 4   P19_POP75P  34980 non-null  float64
 5   siren       34975 non-null  str    
 6   loypredm2   34978 non-null  float64
dtypes: float64(2), int64(2), object(1), str(2)
memory usage: 1.9+ MB


In [83]:
#Création de la variable intermédiaire loyer * pop pour pouvoir disposer d'un loyer moyen à l'échelle EPCI pondéré par la population
df_communes['loyerxpop'] = df_communes['P19_POP']*df_communes['loypredm2']

#Création d'un df à l'échelle EPCI
df_epci = (
    df_communes
    .groupby('siren', as_index=False)
    .agg(
        P19_POP=('P19_POP', 'sum'), #Pour pouvoir avoir aussi la POP75+ % à l'échelle EPCI (p-ê utilisé dans le modèle)
        P19_POP75P=('P19_POP75P', 'sum'),
        nb_communes=('CODGEO', 'count'), # nombre de communes par EPCI
        loyerxpop=('loyerxpop', 'sum')
    )
)
df_communes = df_communes.drop(columns='loyerxpop') #Suppression de la variable intermédiaire dans df_communes
df_epci.head()

,siren,P19_POP,P19_POP75P,nb_communes,loyerxpop
0,200000172,27181,1739.803417,7,351753.608836
1,200000438,35656,2923.468551,9,320617.536383
2,200000545,18593,2034.177122,6,143062.142610
3,200000628,23838,2402.855977,5,225934.400886
4,200000800,10517,1383.161671,6,93010.271183


In [84]:
#Calcul du loyen moyen pondéré
df_epci['loyer_mean_pondere'] = df_epci['loyerxpop'] / df_epci['P19_POP']
df_epci = df_epci.drop(columns='loyerxpop') #Suppression définitive de la variable intérmédiaire

#Calcul du pourcentage de la population ayant 75% +
df_epci['P19_POP75P_perc'] = df_epci['P19_POP75P'] / df_epci['P19_POP'] * 100 

df_epci.head()

,siren,P19_POP,P19_POP75P,nb_communes,loyer_mean_pondere,P19_POP75P_perc
0,200000172,27181,1739.803417,7,12.941158,6.400807
1,200000438,35656,2923.468551,9,8.991966,8.199093
2,200000545,18593,2034.177122,6,7.694409,10.940554
3,200000628,23838,2402.855977,5,9.477909,10.079939
4,200000800,10517,1383.161671,6,8.843803,13.151675


In [85]:
#Import des données relatives au revenu
df_filosofi = pd.read_csv('../data/raw/FILO2019_DISP_EPCI.csv',
                          sep = ";")

print(df_filosofi.columns.tolist())

#Conservation des colonnes qui nous intéresse
df_filosofi = df_filosofi[['CODGEO', 'Q219', 'AGE6Q219', 'AGE6PPAT19', 'AGE6PPSOC19']]

df_filosofi.head()

['CODGEO', 'NBMEN19', 'NBPERS19', 'NBUC19', 'Q119', 'Q219', 'Q319', 'Q3_Q1', 'D119', 'D219', 'D319', 'D419', 'D619', 'D719', 'D819', 'D919', 'RD', 'S80S2019', 'GI19', 'PACT19', 'PTSA19', 'PCHO19', 'PBEN19', 'PPEN19', 'PPAT19', 'PPSOC19', 'PPFAM19', 'PPMINI19', 'PPLOGT19', 'PIMPOT19', 'AGE1Q119', 'AGE1Q219', 'AGE1Q319', 'AGE1Q3_Q1', 'AGE1D119', 'AGE1D219', 'AGE1D319', 'AGE1D419', 'AGE1D619', 'AGE1D719', 'AGE1D819', 'AGE1D919', 'AGE1RD', 'AGE1S80S2019', 'AGE1GI19', 'AGE1PACT19', 'AGE1PTSA19', 'AGE1PCHO19', 'AGE1PBEN19', 'AGE1PPEN19', 'AGE1PPAT19', 'AGE1PPSOC19', 'AGE1PPFAM19', 'AGE1PPMINI19', 'AGE1PPLOGT19', 'AGE1PIMPOT19', 'AGE2Q119', 'AGE2Q219', 'AGE2Q319', 'AGE2Q3_Q1', 'AGE2D119', 'AGE2D219', 'AGE2D319', 'AGE2D419', 'AGE2D619', 'AGE2D719', 'AGE2D819', 'AGE2D919', 'AGE2RD', 'AGE2S80S2019', 'AGE2GI19', 'AGE2PACT19', 'AGE2PTSA19', 'AGE2PCHO19', 'AGE2PBEN19', 'AGE2PPEN19', 'AGE2PPAT19', 'AGE2PPSOC19', 'AGE2PPFAM19', 'AGE2PPMINI19', 'AGE2PPLOGT19', 'AGE2PIMPOT19', 'AGE3Q119', 'AGE3Q219', '

,CODGEO,Q219,AGE6Q219,AGE6PPAT19,AGE6PPSOC19
0,200000172,25520,23360,20.0,1.2
1,200000438,21700,19680,11.2,1.4
2,200000545,18120,20680,10.3,2.2
3,200000628,19770,20600,13.4,2.4
4,200000800,21640,21880,16.0,0.8


In [86]:
df_filosofi.info()

<class 'pandas.DataFrame'>
RangeIndex: 1240 entries, 0 to 1239
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CODGEO       1240 non-null   int64  
 1   Q219         1240 non-null   int64  
 2   AGE6Q219     1240 non-null   int64  
 3   AGE6PPAT19   1240 non-null   float64
 4   AGE6PPSOC19  1240 non-null   float64
dtypes: float64(2), int64(3)
memory usage: 48.6 KB


In [87]:
#Transformation des codes géo en str pour comparaison
df_filosofi['CODGEO'] = df_filosofi['CODGEO'].astype(str)

In [88]:
#Jointure des données filosofi sur le df_epci

df_epci = pd.merge(
    left = df_epci,
    right = df_filosofi,
    left_on = 'siren',
    right_on = 'CODGEO',
    how = 'left',
    validate= 'one_to_one',
)
df_epci = df_epci.drop(columns='CODGEO') #Suppression de la colonne de jointure côté droit

df_epci.head()

,siren,P19_POP,P19_POP75P,nb_communes,loyer_mean_pondere,P19_POP75P_perc,Q219,AGE6Q219,AGE6PPAT19,AGE6PPSOC19
0,200000172,27181,1739.803417,7,12.941158,6.400807,25520.0,23360.0,20.0,1.2
1,200000438,35656,2923.468551,9,8.991966,8.199093,21700.0,19680.0,11.2,1.4
2,200000545,18593,2034.177122,6,7.694409,10.940554,18120.0,20680.0,10.3,2.2
3,200000628,23838,2402.855977,5,9.477909,10.079939,19770.0,20600.0,13.4,2.4
4,200000800,10517,1383.161671,6,8.843803,13.151675,21640.0,21880.0,16.0,0.8


In [89]:
#Comptage du nombre de colonnes vides
len(df_epci[df_epci['Q219'].isna()])

10

In [90]:
#Affichage des EPCI concernés
df_epci[df_epci['Q219'].isna()]

,siren,P19_POP,P19_POP75P,nb_communes,loyer_mean_pondere,P19_POP75P_perc,Q219,AGE6Q219,AGE6PPAT19,AGE6PPSOC19
29,200018653,99532,7691.381949,3,13.860728,7.727547,NaN,NaN,NaN,NaN
50,200027548,29773,452.481691,4,14.369279,1.519772,NaN,NaN,NaN,NaN
201,200041507,63748,4956.921985,4,14.879163,7.775808,NaN,NaN,NaN,NaN
264,200044691,56466,5112.350693,5,13.163133,9.053857,NaN,NaN,NaN,NaN
1237,249710047,10565,1279.252757,3,12.945088,12.108403,NaN,NaN,NaN,NaN
1238,249710062,76742,5550.401084,6,13.697612,7.232547,NaN,NaN,NaN,NaN
1239,249710070,77186,7902.835082,11,13.138474,10.238690,NaN,NaN,NaN,NaN
1242,249730037,96757,906.801248,8,12.784215,0.937194,NaN,NaN,NaN,NaN
1243,249730045,147943,3876.929665,6,14.785300,2.620556,NaN,NaN,NaN,NaN
1244,249730052,7205,81.491048,4,11.718684,1.131035,NaN,NaN,NaN,NaN


>Les 10 EPCI ne diposant pas de données dans filosofi correspondant à la Guyane et la Guadeloupe. 

>Je les exclus en attendant de voir si les revenus jouent un rôle majeur dans le modèle, je les gardes sous forme de liste pour voir si les EHPAD de ces territoires sont référencés
>sur FINESS

In [91]:
#Conservation d'une trace des epci exclus du modèle
epci_exclus = df_epci[df_epci['Q219'].isna()]['siren'].tolist()

In [92]:
# On ajoute le suffixe _epci à toutes les colonnes, sauf siren, qui sert de clé de jointure.
df_epci = df_epci.rename(columns={col: f"{col}_epci" for col in df_epci.columns if col != 'siren'})

# Vérification des nouveaux noms
print(df_epci.columns.tolist())

['siren', 'P19_POP_epci', 'P19_POP75P_epci', 'nb_communes_epci', 'loyer_mean_pondere_epci', 'P19_POP75P_perc_epci', 'Q219_epci', 'AGE6Q219_epci', 'AGE6PPAT19_epci', 'AGE6PPSOC19_epci']


>etablab_stock_et_2019.csv
>* Fichier pour jointure sur la clé nofinesset
>* Conservation temporaire : nofinesset
>* Conservation définitive : cog (équivalent de CODGEO sur base-cc-evolution-structure)
>* Possibilité d'amélioration : coordx, coordy ET les informations dans la colonne liblongmft (en attendant gard ash)

In [93]:
#Import du premier jeu de données avec établissements  FINESS
df_ehpad_geo = pd.read_csv('../data/raw/etalab_stock_et_20191231.csv',
                           sep = ';',
                           encoding='latin-1'
                           )

print(df_ehpad_geo.columns.tolist())

['nofinesset', 'nofinessej', 'rs', 'rslongue', 'complrs', 'compldistrib', 'numvoie', 'typvoie', 'voie', 'compvoie', 'lieuditbp', 'region', 'libregion', 'departement', 'libdepartement', 'cog', 'codepostal', 'libelle_routage', 'ligneacheminement', 'telephone', 'telecopie', 'categetab', 'libcategetab', 'liblongcategetab', 'categretab', 'libcategretab', 'siret', 'codeape', 'libcodeape', 'mft', 'libmft', 'liblongmft', 'sph', 'libsph', 'numen', 'coordx', 'coordy', 'sourcegeocod', 'dategeocod', 'dateautor', 'dateouvert', 'datemaj']


In [94]:
#Conservation des colonnes utiles
df_ehpad_geo = df_ehpad_geo[['nofinesset', 'nofinessej', 'libcategetab','cog', 'liblongmft','coordx', 'coordy']]
df_ehpad_geo.head()

,nofinesset,nofinessej,libcategetab,cog,liblongmft,coordx,coordy
0,040002313,040001646,Pharmacie d'Officine,04110,Etablissement Tarif Libre,"950,414.1","6,331,695.3"
1,040000069,040780181,C.H. (ex H.L.),04088,ARS établissements Publics de santé dotation g...,"923,147.4","6,321,804"
2,040002891,040002214,Pharmacie d'Officine,04209,Etablissement Tarif Libre,"935,097","6,348,465.9"
3,040002362,040001695,Pharmacie d'Officine,04209,Etablissement Tarif Libre,"935,420.8","6,347,610.1"
4,040002750,040002073,Pharmacie d'Officine,04112,Etablissement Tarif Libre,"923,955.3","6,307,570.5"


In [95]:
#Nettoyage des catégories hors EHPAD
df_ehpad_geo = df_ehpad_geo[df_ehpad_geo['libcategetab'] == "EHPAD"]
df_ehpad_geo.head()

,nofinesset,nofinessej,libcategetab,cog,liblongmft,coordx,coordy
12,040785925,040780231,EHPAD,04166,"ARS/PCD, Tarif partiel, habilité aide sociale,...","948,625.8","6,307,051.1"
14,040785776,040004913,EHPAD,04096,"ARS/PCD, Tarif partiel, habilité aide sociale ...","996,682.1","6,375,669.7"
52,050003029,050002989,EHPAD,05061,"ARS/PCD, Tarif global, habilité aide sociale s...","943,968.6","6,389,093"
80,030780605,030000236,EHPAD,03187,"ARS/PCD, Tarif global, habilité aide sociale s...","734,516.1","6,581,532"
81,030780969,030000368,EHPAD,03108,"ARS/PCD, Tarif global, habilité aide sociale s...","694,957.9","6,564,793.5"


In [96]:
len(df_ehpad_geo) #Vérification du nb d'ehpad

7520

In [97]:
df_ehpad_geo.info()

<class 'pandas.DataFrame'>
Index: 7520 entries, 12 to 94881
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   nofinesset    7520 non-null   str  
 1   nofinessej    7520 non-null   str  
 2   libcategetab  7520 non-null   str  
 3   cog           7520 non-null   str  
 4   liblongmft    7520 non-null   str  
 5   coordx        7520 non-null   str  
 6   coordy        7520 non-null   str  
dtypes: str(7)
memory usage: 470.0 KB


In [98]:
#Vérification du compte d'établissement et de groupes
print(df_ehpad_geo['nofinesset'].nunique())
print(df_ehpad_geo['nofinessej'].nunique())

7520
4912


>cnsa-export-prix-ehpad2019.xlsx

>* Conservation temporaire pour jointure : finesset
>* il indique que finesset est le numéro juridique et finessej le numéro géographique. C'est l’inverse
>* Concentration sur l'hébergement permanent (conseil de Cyril) + en chambre seule -> Tous les ehpad ne proposent pas de chmbre double et tous les ehpad ne proposent pas d'hébergement temporaire : permanent seul colonne rempli à \~100%
>* A vu d'oeil certains ehpad n'ont pas renseignés leur nombre de place mais ont renseigné leur tarif -> à virer du modèle et voir cb d'ehpad ça exclut
>* PRIX_MOIS_HEBPERM_CS_GIR56 -> "prix de base" pour un résident avec une faible perte d'autonomie
>* Conservation temporaire pour jointure : finesset (ou définitive et utilisation de cette colonne comme primary key)
>* Conservation définitive : STATUT_JUR, ASH, nb_capins_HP, pPRIX_MOIS\HEBPERM_CS_GIR56
>* Hésitation : finessej pour regrouper les établissements par gestionnaire. Rajoute du bruit pour rien ? **(QUESTION A POSER)** Faire des stats sur finessej pour voir les différences de prix entre établissements du même gestionnaire

In [99]:
#Importation du jeu de données qui sera le principal

df_ehpad = pd.read_excel('../data/raw/cnsa-export-prix-ehpad-2019-retraitee.xlsx',
                         sheet_name = 'cnsa_export_prix_ehpad_retraité')

print(df_ehpad.columns.tolist())

['Année', 'finesset', 'finessej', 'Catégorie', 'STATUT_JUR', 'ASH', 'nb_capins_HP', 'nb_capins_HT', 'prixHebPermCs', 'prixHebPermCd', 'prixHebPermCsa', 'prixHebPermCda', 'prixHebTempCs', 'prixHebTempCd', 'prixHebTempCsa', 'prixHebTempCda', 'tarifGir12', 'tarifGir34', 'tarifGir56', 'PRIX_HEB_PERM_CS', 'PRIX_HEB_PERM_CD', 'PRIX_HEB_PERM_CSA', 'PRIX_HEB_PERM_CDA', 'PRIX_HEB_TEMP_CS', 'PRIX_HEB_TEMP_CD', 'PRIX_HEB_TEMP_CSA', 'PRIX_HEB_TEMP_CDA', 'TARIF_GIR_12', 'TARIF_GIR_34', 'TARIF_GIR_56', 'PRIX_MOIS_HEBPERM_CS_GIR56']


In [100]:
#Conservation des colonnes essentielles
df_ehpad = df_ehpad[['finesset', 'finessej', 'STATUT_JUR', 'ASH', 'nb_capins_HP', 'PRIX_MOIS_HEBPERM_CS_GIR56']]
df_ehpad.head()

,finesset,finessej,STATUT_JUR,ASH,nb_capins_HP,PRIX_MOIS_HEBPERM_CS_GIR56
0,010002228,750057622,3-Privé commercial,NON,60,3154.8
1,010004059,010783009,2-Privé non lucratif,OUI,66,2204.4
2,010006799,690802715,2-Privé non lucratif,OUI,80,2092.2
3,010008571,010007987,1-Public,OUI,69,1561.5
4,010009223,010787109,2-Privé non lucratif,OUI,75,2189.7


In [101]:
#Vérification du compte d'établissement et de groupes
print(df_ehpad['finesset'].nunique())
print(df_ehpad['finessej'].nunique())

6751
4496


In [102]:
#Jointure des deux jeux de données concernant des informations établissements, direction sur le df_ehpad qui sera le principal
df_ehpad = pd.merge(left= df_ehpad,
                    right= df_ehpad_geo,
                    left_on = 'finesset',
                    right_on = 'nofinesset',
                    how = 'left'
)

df_ehpad = df_ehpad.drop(columns=['nofinesset', 'nofinessej']) #Suppression des colonnes doublons

df_ehpad.head()

,finesset,finessej,STATUT_JUR,ASH,nb_capins_HP,PRIX_MOIS_HEBPERM_CS_GIR56,libcategetab,cog,liblongmft,coordx,coordy
0,010002228,750057622,3-Privé commercial,NON,60,3154.8,EHPAD,01004,"ARS/PCD, Tarif partiel, non habilité aide soci...","881,334.9","6,542,456.9"
1,010004059,010783009,2-Privé non lucratif,OUI,66,2204.4,EHPAD,01281,"ARS/PCD, Tarif global, habilité aide sociale s...","939,632.3","6,578,432"
2,010006799,690802715,2-Privé non lucratif,OUI,80,2092.2,EHPAD,01194,"ARS/PCD, Tarif partiel, habilité aide sociale ...","836,009.5","6,544,033.9"
3,010008571,010007987,1-Public,OUI,69,1561.5,EHPAD,01185,"ARS/PCD, Tarif global, habilité aide sociale, ...","902,289.9","6,545,912.8"
4,010009223,010787109,2-Privé non lucratif,OUI,75,2189.7,EHPAD,01181,"ARS/PCD, Tarif partiel, habilité aide sociale ...","900,979.5","6,572,377.3"


In [103]:
#Vérification des valeurs manquantes dans chaque colonne
for col in df_ehpad.columns:
    mask_na = df_ehpad[col].isna()
    nb_na = mask_na.sum()
    if nb_na > 0: #Affichage uniquement si le df n'est pas vide
        print(f"Nom de la colonne : {col} et nombre de valeurs manquantes : {nb_na}")
        display(df_ehpad[mask_na])
        # On ne supprime que les établissements sans prix (variable cible) 
        # les valeurs manquantes des autres colonnes sont seulement affichées
        if col == 'PRIX_MOIS_HEBPERM_CS_GIR56':
            finess_to_delete.extend(df_ehpad.loc[mask_na, 'finesset'].tolist())  # Ajout des établissements à la liste des finess à supprimer

finess_to_delete = list(set(finess_to_delete)) #Set pour éviter les doublons

print(f"Nombres d'établissements supprimés au total : {len(finess_to_delete)}")




Nom de la colonne : PRIX_MOIS_HEBPERM_CS_GIR56 et nombre de valeurs manquantes : 29


,finesset,finessej,STATUT_JUR,ASH,nb_capins_HP,PRIX_MOIS_HEBPERM_CS_GIR56,libcategetab,cog,liblongmft,coordx,coordy
255,060022902,060785011,1-Public,OUI,40,NaN,EHPAD,06088,"ARS/PCD, Tarif global, habilité aide sociale, ...","1,044,263.3","6,300,647.7"
296,060790789,060785011,1-Public,OUI,83,NaN,EHPAD,06163,"ARS/PCD, Tarif global, habilité aide sociale, ...","1,067,384.5","6,339,688.7"
647,130033178,130781446,1-Public,OUI,30,NaN,EHPAD,13005,"ARS/PCD, Tarif global, habilité aide sociale, ...","908,116.8","6,246,972"
775,130809940,130007198,2-Privé non lucratif,NON,0,NaN,EHPAD,13005,"ARS/PCD, Tarif partiel, non habilité aide soci...","906,847.6","6,246,065.5"
1318,220014823,220014666,2-Privé non lucratif,OUI,0,NaN,EHPAD,22223,"ARS/PCD, Tarif global, habilité aide sociale s...","244,222.1","6,848,836.2"
1680,290008846,290007335,2-Privé non lucratif,OUI,0,NaN,EHPAD,29075,"ARS/PCD, Tarif partiel, habilité aide sociale ...","153,273.5","6,840,434.5"
1722,290025444,290007053,1-Public,OUI,0,NaN,EHPAD,29019,"ARS/PCD, Tarif partiel, habilité aide sociale ...","145,244.7","6,836,266.3"
1736,290035047,290000546,2-Privé non lucratif,OUI,0,NaN,EHPAD,29105,"ARS/PCD, Tarif global, habilité aide sociale, ...","178,655.8","6,845,947.5"
1756,2B0003628,2B0000020,1-Public,OUI,38,NaN,EHPAD,2B033,"ARS/PCD, Tarif global, habilité aide sociale s...","1,228,475.4","6,200,115.9"
1862,310019104,310019096,3-Privé commercial,OUI,80,NaN,EHPAD,31555,"ARS/PCD, Tarif global, habilité aide sociale s...","577,198.7","6,278,043"


Nombres d'établissements supprimés au total : 33


In [104]:
nb_avant = len(df_ehpad)

# On garde uniquement les établissements dont le finesset N'EST PAS dans la liste à supprimer
# ~ inverse la sélection, et reset_index(drop=True) renumérote les lignes de 0 à n-1
df_ehpad = df_ehpad[~df_ehpad['finesset'].isin(finess_to_delete)].reset_index(drop=True)

print(f"{nb_avant - len(df_ehpad)} EHPAD supprimés, {len(df_ehpad)} restants") #Si moins, c'est qu'un des EHPAD pré-identifiés à supprimer ne figurait pas dans le fichier

31 EHPAD supprimés, 6720 restants


In [105]:
#Jointure du df_ehpad avec le df_communes

df_ehpad = pd.merge(left=df_ehpad,
                    right = df_communes,
                    left_on = 'cog',
                    right_on = 'CODGEO',
                    how = 'left',
                    validate='many_to_one').drop(columns='CODGEO') #Plusieurs ehpads dans la même commune
df_ehpad.head()

,finesset,finessej,STATUT_JUR,ASH,nb_capins_HP,PRIX_MOIS_HEBPERM_CS_GIR56,libcategetab,cog,liblongmft,coordx,coordy,REG,DEP,P19_POP,P19_POP75P,siren,loypredm2
0,010002228,750057622,3-Privé commercial,NON,60,3154.8,EHPAD,01004,"ARS/PCD, Tarif partiel, non habilité aide soci...","881,334.9","6,542,456.9",84.0,01,14134.0,1239.164308,240100883,10.074507
1,010004059,010783009,2-Privé non lucratif,OUI,66,2204.4,EHPAD,01281,"ARS/PCD, Tarif global, habilité aide sociale s...","939,632.3","6,578,432",84.0,01,4483.0,230.557715,240100750,20.693334
2,010006799,690802715,2-Privé non lucratif,OUI,80,2092.2,EHPAD,01194,"ARS/PCD, Tarif partiel, habilité aide sociale ...","836,009.5","6,544,033.9",84.0,01,6346.0,680.405938,200040590,10.984294
3,010008571,010007987,1-Public,OUI,69,1561.5,EHPAD,01185,"ARS/PCD, Tarif global, habilité aide sociale, ...","902,289.9","6,545,912.8",84.0,01,4844.0,765.384953,200042935,7.996378
4,010009223,010787109,2-Privé non lucratif,OUI,75,2189.7,EHPAD,01181,"ARS/PCD, Tarif partiel, habilité aide sociale ...","900,979.5","6,572,377.3",84.0,01,1241.0,138.000000,200042935,8.931485


In [106]:
#Identification des communes sans sirene
print(len(df_ehpad[df_ehpad['siren'].isna()]))

print(df_ehpad[df_ehpad['siren'].isna()]['cog'].tolist())

47
['27676', '9A101', '9A122', '9A117', '9A126', '9A128', '9A115', '9A124', '9A117', '9A116', '9A123', '9A113', '9A117', '9B222', '9B210', '9B221', '9B229', '9B202', '9B217', '9B223', '9B231', '9B230', '9B204', '9B213', '9B229', '9B207', '9B223', '9B209', '9B212', '9B219', '9B209', '9C302', '9C311', '9D415', '9D408', '9D415', '9D411', '9D410', '9D403', '9D416', '9D416', '9D416', '9D409', '9D411', '9D415', '9D411', '9D411']


>Les communes concernés sont les DOM TOM, je concentre mon modèle sur la france métropolitaine

In [107]:
nb_avant = len(df_ehpad)

df_ehpad = df_ehpad.dropna(subset=['siren']).reset_index(drop=True)

print(f"{nb_avant - len(df_ehpad)} EHPAD supprimés, {len(df_ehpad)} restants")
print(df_ehpad['siren'].isna().sum())  # attendu : 0

47 EHPAD supprimés, 6673 restants
0


In [108]:
df_ehpad.info()

<class 'pandas.DataFrame'>
RangeIndex: 6673 entries, 0 to 6672
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   finesset                    6673 non-null   str    
 1   finessej                    6673 non-null   str    
 2   STATUT_JUR                  6673 non-null   str    
 3   ASH                         6673 non-null   str    
 4   nb_capins_HP                6673 non-null   int64  
 5   PRIX_MOIS_HEBPERM_CS_GIR56  6673 non-null   float64
 6   libcategetab                6673 non-null   str    
 7   cog                         6673 non-null   str    
 8   liblongmft                  6673 non-null   str    
 9   coordx                      6673 non-null   str    
 10  coordy                      6673 non-null   str    
 11  REG                         6673 non-null   float64
 12  DEP                         6673 non-null   object 
 13  P19_POP                     6673 non-null   

In [109]:
#Création d'un df pour calculer le nombre d'établissement par groupe (peut influencer sur le prix)
df_ehpad_groupe = df_ehpad.groupby('finessej', as_index=False).agg(nb_etab_grp=('finesset', 'count'))
df_ehpad_groupe.sort_values(by='nb_etab_grp', ascending=False) #Desc pour éviter de ne voir que les groupe à 1 établissements

,finessej,nb_etab_grp
4328,920030152,174
3503,750056335,104
4322,920028560,62
2544,570010173,55
1431,330050899,38
...,...,...
4442,950040071,1
396,120004718,1
4444,950110049,1
4445,950150037,1


In [110]:
# Jointure pour récupérer l'information du nb d'établissements par groupe
df_ehpad = pd.merge(left = df_ehpad,
                    right = df_ehpad_groupe,
                    on='finessej',
                    how='left',
                    validate='many_to_one')

>Limites : 
>- Je compte sur un df déjà réduit. J'ai pu déjà supprimé des EHPAD appartenant à un groupe, le réduisant mécaniquement
>- finessej désigne l’entité juridique, pas forcément le groupe. Certains grands groupe crée une société par établissement ! 

In [111]:
df_ehpad.info()

<class 'pandas.DataFrame'>
RangeIndex: 6673 entries, 0 to 6672
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   finesset                    6673 non-null   str    
 1   finessej                    6673 non-null   str    
 2   STATUT_JUR                  6673 non-null   str    
 3   ASH                         6673 non-null   str    
 4   nb_capins_HP                6673 non-null   int64  
 5   PRIX_MOIS_HEBPERM_CS_GIR56  6673 non-null   float64
 6   libcategetab                6673 non-null   str    
 7   cog                         6673 non-null   str    
 8   liblongmft                  6673 non-null   str    
 9   coordx                      6673 non-null   str    
 10  coordy                      6673 non-null   str    
 11  REG                         6673 non-null   float64
 12  DEP                         6673 non-null   object 
 13  P19_POP                     6673 non-null   

In [112]:
df_ehpad = pd.merge(left=df_ehpad,
                    right = df_epci,
                    on='siren',
                    how = 'left',
                    validate = 'many_to_one')
df_ehpad

,finesset,finessej,STATUT_JUR,ASH,nb_capins_HP,PRIX_MOIS_HEBPERM_CS_GIR56,libcategetab,cog,liblongmft,coordx,...,nb_etab_grp,P19_POP_epci,P19_POP75P_epci,nb_communes_epci,loyer_mean_pondere_epci,P19_POP75P_perc_epci,Q219_epci,AGE6Q219_epci,AGE6PPAT19_epci,AGE6PPSOC19_epci
0,010002228,750057622,3-Privé commercial,NON,60,3154.8,EHPAD,01004,"ARS/PCD, Tarif partiel, non habilité aide soci...","881,334.9",...,1,79063,6079.193731,53,9.810075,7.689050,23000.0,22130.0,16.2,1.3
1,010004059,010783009,2-Privé non lucratif,OUI,66,2204.4,EHPAD,01281,"ARS/PCD, Tarif global, habilité aide sociale s...","939,632.3",...,4,98257,4730.848070,27,19.321257,4.814770,36370.0,31890.0,18.3,0.8
2,010006799,690802715,2-Privé non lucratif,OUI,80,2092.2,EHPAD,01194,"ARS/PCD, Tarif partiel, habilité aide sociale ...","836,009.5",...,26,72815,6745.075392,18,10.212145,9.263305,21240.0,22010.0,18.5,1.8
3,010008571,010007987,1-Public,OUI,69,1561.5,EHPAD,01185,"ARS/PCD, Tarif global, habilité aide sociale, ...","902,289.9",...,1,63365,5266.837161,42,8.814524,8.311903,20470.0,21500.0,16.9,1.7
4,010009223,010787109,2-Privé non lucratif,OUI,75,2189.7,EHPAD,01181,"ARS/PCD, Tarif partiel, habilité aide sociale ...","900,979.5",...,2,63365,5266.837161,42,8.814524,8.311903,20470.0,21500.0,16.9,1.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6668,950807826,920031267,3-Privé commercial,NON,90,2843.4,EHPAD,95229,"ARS/PCD, Tarif global, non habilité aide socia...","653,446.8",...,1,183125,13998.032092,18,16.846945,7.643977,24990.0,26550.0,19.1,1.2
6669,950808469,950014738,3-Privé commercial,NON,60,2178.3,EHPAD,95142,"ARS/PCD, Tarif partiel, non habilité aide soci...","622,526",...,3,24767,1540.614536,34,13.503282,6.220433,27340.0,26730.0,21.6,0.5
6670,950808519,950808501,3-Privé commercial,NON,85,2974.8,EHPAD,95572,"ARS/PCD, Tarif partiel, non habilité aide soci...","634,913.9",...,1,212398,8073.990327,13,16.088356,3.801350,22100.0,23780.0,11.7,2.2
6671,950808956,250015658,3-Privé commercial,NON,103,3137.4,EHPAD,95203,"ARS/PCD, Tarif partiel, non habilité aide soci...","647,197.5",...,25,282028,19390.408016,15,16.335829,6.875349,24760.0,25920.0,14.3,1.2


In [113]:
print(df_ehpad.columns.tolist())

['finesset', 'finessej', 'STATUT_JUR', 'ASH', 'nb_capins_HP', 'PRIX_MOIS_HEBPERM_CS_GIR56', 'libcategetab', 'cog', 'liblongmft', 'coordx', 'coordy', 'REG', 'DEP', 'P19_POP', 'P19_POP75P', 'siren', 'loypredm2', 'nb_etab_grp', 'P19_POP_epci', 'P19_POP75P_epci', 'nb_communes_epci', 'loyer_mean_pondere_epci', 'P19_POP75P_perc_epci', 'Q219_epci', 'AGE6Q219_epci', 'AGE6PPAT19_epci', 'AGE6PPSOC19_epci']


In [114]:
df_ehpad['P19_POP75P_perc'] = df_ehpad['P19_POP75P'] / df_ehpad['P19_POP'] * 100

In [115]:
#Réorganisation des colonnes pour plus de lisibilité : 

df_ehpad = df_ehpad[[
    # Identifiants
    'finesset', 'finessej',
    # Variable cible
    'PRIX_MOIS_HEBPERM_CS_GIR56',
    # Caractéristiques de l'établissement
    'STATUT_JUR', 'ASH', 'nb_capins_HP', 'nb_etab_grp', 'liblongmft',
    # Territoire : commune
    'P19_POP', 'P19_POP75P', 'P19_POP75P_perc', 'loypredm2',
    # Territoire : EPCI
    'nb_communes_epci', 'P19_POP_epci', 'P19_POP75P_epci', 'P19_POP75P_perc_epci',
    'Q219_epci', 'AGE6Q219_epci', 'AGE6PPAT19_epci', 'AGE6PPSOC19_epci',
    'loyer_mean_pondere_epci',
    # Localisation
    'REG', 'DEP', 'siren', 'cog', 'coordx', 'coordy'
]]

df_ehpad.head(2)

,finesset,finessej,PRIX_MOIS_HEBPERM_CS_GIR56,STATUT_JUR,ASH,nb_capins_HP,nb_etab_grp,liblongmft,P19_POP,P19_POP75P,...,AGE6Q219_epci,AGE6PPAT19_epci,AGE6PPSOC19_epci,loyer_mean_pondere_epci,REG,DEP,siren,cog,coordx,coordy
0,010002228,750057622,3154.8,3-Privé commercial,NON,60,1,"ARS/PCD, Tarif partiel, non habilité aide soci...",14134.0,1239.164308,...,22130.0,16.2,1.3,9.810075,84.0,01,240100883,01004,"881,334.9","6,542,456.9"
1,010004059,010783009,2204.4,2-Privé non lucratif,OUI,66,4,"ARS/PCD, Tarif global, habilité aide sociale s...",4483.0,230.557715,...,31890.0,18.3,0.8,19.321257,84.0,01,240100750,01281,"939,632.3","6,578,432"


In [116]:
# Corrections des datatype

df_ehpad['finesset'] = df_ehpad['finesset'].astype(str)
df_ehpad['finessej'] = df_ehpad['finessej'].astype(str)
df_ehpad['siren'] = df_ehpad['siren'].astype(str)
df_ehpad['cog'] = df_ehpad['cog'].astype(str)
df_ehpad['DEP'] = df_ehpad['DEP'].astype(str)
df_ehpad['REG'] = df_ehpad['REG'].astype(str)
df_ehpad[['nb_capins_HP', 'nb_etab_grp']].astype(int)

#correction sur les coordonnés
df_ehpad["coordx"] = df_ehpad["coordx"].str.replace(",", "").astype(float)
df_ehpad["coordy"] = df_ehpad["coordy"].str.replace(",", "").astype(float)

# Retirer le ".0" en fin du code REG
df_ehpad["REG"] = df_ehpad["REG"].str.replace(".0", "", regex=False)




In [117]:
df_ehpad['siren'].nunique() #Dernière vérification pour voir combien d'EPCI dans notre df

1184

>1184 soit 95% des EPCI de France Métropolitaine

>## Ajout de deux variables : ancienneté et taux d'équipement

>Suite à l'exploration, deux variables sont ajoutées :
>- **`anciennete`** : nombre d'années depuis l'ouverture de l'établissement (FINESS, `dateouvert`). Elle sert à approcher le standing et le coût du bâtiment.
>- **`taux_equip_epci`** : nombre de places d'EHPAD pour 1 000 personnes de 75 ans et plus dans l'EPCI. Elle mesure la concurrence locale.

In [118]:
# Relecture du fichier FINESS : identifiant, catégorie et date d'ouverture
finess_dates = pd.read_csv('../data/raw/etalab_stock_et_20191231.csv',
                           sep=';',                                 # séparateur du fichier FINESS
                           encoding='latin-1',                      # encodage du fichier FINESS
                           usecols=['nofinesset', 'libcategetab', 'dateouvert'],  # on ne lit que les colonnes utiles
                           dtype={'nofinesset': str})               # identifiant en texte pour garder les zéros

# On ne garde que les EHPAD (le fichier contient tous les équipements sociaux et médico-sociaux)
finess_dates = finess_dates[finess_dates['libcategetab'] == "EHPAD"]

# Conversion de la date d'ouverture en année (nombre)
finess_dates['annee_ouverture'] = pd.to_datetime(finess_dates['dateouvert'], errors='coerce').dt.year

# Années les plus fréquentes, pour repérer d'éventuelles dates par défaut
finess_dates['annee_ouverture'].value_counts().head(5)


annee_ouverture
1901    577
1904    326
1990    251
1992    228
1989    221
Name: count, dtype: int64

>Les années **1901** et **1904** reviennent anormalement souvent (plus de 800 EHPAD à elles deux). Ce sont des dates par défaut saisies quand la vraie date est inconnue : on les remplace par des valeurs manquantes.

In [119]:
# Remplacer les dates par défaut (1901 et 1904) par une valeur manquante
finess_dates['annee_ouverture'] = finess_dates['annee_ouverture'].mask(
    finess_dates['annee_ouverture'].isin([1901, 1904])     # condition : l'année vaut 1901 ou 1904
)                                                          # mask() met une valeur manquante là où la condition est vraie

# Ancienneté en 2019, année des données de prix
finess_dates['anciennete'] = 2019 - finess_dates['annee_ouverture']

# Ajout de l'ancienneté à df_ehpad, en reliant finesset (df_ehpad) à nofinesset (FINESS)
df_ehpad = df_ehpad.merge(finess_dates[['nofinesset', 'anciennete']],
                          left_on='finesset', right_on='nofinesset',
                          how='left')                               # on garde tous les EHPAD de df_ehpad

# Suppression de la colonne d'identifiant en double créée par la fusion
df_ehpad = df_ehpad.drop(columns='nofinesset')

# Vérifications : nombre de lignes inchangé et nombre d'anciennetés manquantes
print(df_ehpad.shape)
print("Ancienneté manquante :", df_ehpad['anciennete'].isna().sum())

# Indicateur : garder la trace des anciennetés qui vont être remplacées
df_ehpad['anciennete_imputee'] = df_ehpad['anciennete'].isna()   # True si l'ancienneté était manquante

# Calcul de la médiane de l'ancienneté (les valeurs manquantes sont ignorées)
mediane_anciennete = df_ehpad['anciennete'].median()
print("Médiane utilisée :", mediane_anciennete)

# Remplacement des valeurs manquantes par la médiane
df_ehpad['anciennete'] = df_ehpad['anciennete'].fillna(mediane_anciennete)   # fillna() remplit les cases vides

# Vérification : il ne doit plus rester de valeur manquante
print("Ancienneté manquante après remplacement :", df_ehpad['anciennete'].isna().sum())

(6673, 28)
Ancienneté manquante : 816
Médiane utilisée : 29.0
Ancienneté manquante après remplacement : 0


In [120]:
# Nombre total de places d'EHPAD dans chaque EPCI (somme des places des EHPAD de l'EPCI)
places_epci = df_ehpad.groupby('siren')['nb_capins_HP'].transform('sum')

# Places pour 1 000 personnes de 75 ans et plus dans l'EPCI
df_ehpad['taux_equip_epci'] = places_epci / df_ehpad['P19_POP75P_epci'] * 1000

# Statistiques de la nouvelle variable
df_ehpad['taux_equip_epci'].describe()

count    6673.000000
mean       98.396295
std        39.105883
min        18.842129
25%        69.836446
50%        89.726898
75%       117.364673
max       298.731359
Name: taux_equip_epci, dtype: float64

In [ ]:
# Nouvelle sauvegarde du fichier propre, avec les deux variables ajoutées
df_ehpad.to_csv('../data/clean/ehpad.csv', index = False)